# 03 · Baseline model screening with chronological CV

This notebook runs Phase A of model selection. Every candidate is a complete raw-row pipeline. Chronological folds live inside the training period, candidate thresholds/ranking use validation, and **holdout is not evaluated here**. The saved comparison is the frozen screening handoff consumed by notebook 04.

GPU mode enables CUDA for XGBoost and OpenCL for LightGBM, while retaining the CPU logistic and random-forest baselines. Set `FRAUD_NOTEBOOK_GPU=0` for CPU mode.


## 1. Load prepared data and the selected recipe


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository from either the project root or notebook/.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "src" / "fraud_detection").is_dir():
        PROJECT_ROOT = directory
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected src/fraud_detection."
    )

source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

SEED = 42
SMOKE = os.environ.get("FRAUD_NOTEBOOK_SMOKE", "0") == "1"
MAX_ROWS = int(os.environ.get("FRAUD_NOTEBOOK_MAX_ROWS", "0"))
if MAX_ROWS < 0:
    raise ValueError("FRAUD_NOTEBOOK_MAX_ROWS must be nonnegative.")

stage_setting = os.environ.get(
    "FRAUD_NOTEBOOK_STAGE_DIR", "data/processed/learning_stages"
)
STAGE_ROOT = Path(stage_setting).expanduser()
if not STAGE_ROOT.is_absolute():
    STAGE_ROOT = PROJECT_ROOT / STAGE_ROOT
STAGE_ROOT = STAGE_ROOT.resolve()
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "stage_root": str(STAGE_ROOT),
    "smoke": SMOKE,
    "max_rows": MAX_ROWS,
    "seed": SEED,
})

from fraud_detection.data_preparation import load_prepared_data
from fraud_detection.data import calculate_file_sha256
from fraud_detection.utils import frame_fingerprint

pointer_path = STAGE_ROOT / "prepared_latest.json"
if not pointer_path.is_file():
    raise FileNotFoundError(
        f"Missing {pointer_path}. Run notebook 00_data_preparation.ipynb first."
    )
pointer = json.loads(pointer_path.read_text(encoding="utf-8"))

if bool(pointer.get("smoke")) != SMOKE:
    raise ValueError(
        "Notebook mode differs from notebook 00. Use the same FRAUD_NOTEBOOK_SMOKE setting."
    )
if int(pointer.get("max_rows", 0)) != MAX_ROWS:
    raise ValueError(
        "FRAUD_NOTEBOOK_MAX_ROWS differs from notebook 00. Use the same row limit."
    )

prepared_folder = Path(pointer["prepared_folder"])
prepared_data = load_prepared_data(prepared_folder)
manifest_path = prepared_folder / "prepared_manifest.json"
actual_manifest_sha256 = calculate_file_sha256(manifest_path)
if actual_manifest_sha256 != pointer["prepared_manifest_sha256"]:
    raise ValueError("Prepared manifest changed since notebook 00.")

partitions = prepared_data["partitions"]
labeled_pool = prepared_data["labeled_pool"]
kaggle_test = prepared_data["kaggle_test"]
DATA_SCOPE = prepared_data["manifest"]["data_scope"]
if DATA_SCOPE != pointer["data_scope"]:
    raise ValueError("Prepared data scope differs from notebook 00 pointer.")

current_fingerprints = {
    name: frame_fingerprint(getattr(partitions, name))
    for name in ("train", "validation", "holdout")
}
if current_fingerprints != pointer["partition_fingerprints"]:
    raise ValueError("Prepared partition fingerprints differ from notebook 00.")

SOURCE_FILE_SHA256 = dict(pointer.get("source_file_sha256", {}))
JOIN_QUALITY_REPORT = prepared_data.get("quality_report", {})

print({
    "data_scope": DATA_SCOPE,
    "prepared_folder": str(prepared_folder),
    "train_rows": len(partitions.train),
    "validation_rows": len(partitions.validation),
    "holdout_rows": len(partitions.holdout),
    "kaggle_test_rows": len(kaggle_test),
    "integrity": "checksum + shape + partition fingerprints verified",
})

model_recipe_path = STAGE_ROOT / "model_recipe.json"
if not model_recipe_path.is_file():
    raise FileNotFoundError("Run 02_feature_selection.ipynb first.")
model_recipe = json.loads(model_recipe_path.read_text(encoding="utf-8"))
if model_recipe["prepared_manifest_sha256"] != pointer["prepared_manifest_sha256"]:
    raise ValueError("Model recipe belongs to a different prepared dataset.")
if model_recipe["partition_fingerprints"] != pointer["partition_fingerprints"]:
    raise ValueError("Model recipe partition lineage differs.")


## 2. Build complete candidate configurations


In [ ]:
from copy import deepcopy
from fraud_detection.experiments import (
    compare_candidates,
    default_candidates,
    save_comparison,
    shortlist_candidates,
)

CV_FOLDS = int(os.environ.get("FRAUD_NOTEBOOK_CV_FOLDS", "2"))
if CV_FOLDS < 2:
    raise ValueError("FRAUD_NOTEBOOK_CV_FOLDS must be at least 2")

recipe_keys = (
    "encoding",
    "selection_method",
    "selection_device",
    "selection_max_features",
    "feature_groups",
    "n_clusters",
    "raw_screen_strategy",
    "top_v_features",
    "preserve_features",
    "selection_consensus_components",
)
# Recipes saved before GPU support used CPU selection.
if "selection_device" not in model_recipe:
    model_recipe["selection_device"] = "cpu"
    print("Loaded an earlier CPU feature-selection recipe. Rerun updated notebook 02 for GPU LightGBM selection.")
feature_policy = {key: deepcopy(model_recipe[key]) for key in recipe_keys}

GPU_ENABLED = os.environ.get("FRAUD_NOTEBOOK_GPU", "1") == "1"
candidates = default_candidates(smoke=SMOKE, gpu=GPU_ENABLED)
print({"gpu_enabled": GPU_ENABLED, "xgboost_device": "cuda" if GPU_ENABLED else "cpu", "lightgbm_device": "gpu" if GPU_ENABLED else "cpu"})
for config in candidates.values():
    config.update(deepcopy(feature_policy))

display(pd.DataFrame([
    {"candidate": name, **{k:v for k,v in cfg.items() if k != "model_params"}, "model_params": cfg.get("model_params", {})}
    for name, cfg in candidates.items()
]))


## 3. Phase A: untuned chronological screening


In [ ]:
comparison_result = compare_candidates(
    partitions,
    candidates=candidates,
    seed=SEED,
    n_splits=CV_FOLDS,
    tune=False,
    evaluate_holdout=False,
    data_scope=DATA_SCOPE,
)

comparison_result.metadata.update(
    feature_recipe=feature_policy,
    prepared_manifest_sha256=pointer["prepared_manifest_sha256"],
    source_file_sha256=SOURCE_FILE_SHA256,
    join_quality_report=JOIN_QUALITY_REPORT,
)
for candidate_result in comparison_result.results.values():
    candidate_result.metadata.update(
        source_file_sha256=SOURCE_FILE_SHA256,
        join_quality_report=JOIN_QUALITY_REPORT,
    )

display(comparison_result.table)
print("skipped optional candidates:", comparison_result.skipped)
assert not comparison_result.metadata["holdout_evaluated"]
assert all("holdout" not in r.metrics for r in comparison_result.results.values())


## 4. Inspect the training-CV shortlist


In [ ]:
shortlist_size = min(3, len(comparison_result.results))
shortlist = shortlist_candidates(comparison_result, shortlist_size=shortlist_size)
print({
    "selection_rule": "training chronological CV average precision",
    "shortlist": shortlist,
    "validation_table_winner": comparison_result.winner_name,
    "holdout_evaluated": False,
})


## 5. Save the baseline comparison handoff


In [ ]:
baseline_root = STAGE_ROOT / "model_comparisons"
baseline_root.mkdir(parents=True, exist_ok=True)
baseline_folder = save_comparison(comparison_result, output_dir=baseline_root)

baseline_pointer = {
    "schema_version": 1,
    "comparison_folder": str(baseline_folder.resolve()),
    "comparison_id": comparison_result.metadata["comparison_id"],
    "data_scope": DATA_SCOPE,
    "prepared_manifest_sha256": pointer["prepared_manifest_sha256"],
    "model_recipe_path": str(model_recipe_path.resolve()),
    "cv_folds": CV_FOLDS,
    "winner_name": comparison_result.winner_name,
    "winner_artifact_id": comparison_result.winner.metadata["artifact_id"],
    "holdout_evaluated": False,
}
baseline_pointer_path = STAGE_ROOT / "baseline_latest.json"
baseline_pointer_path.write_text(json.dumps(baseline_pointer, indent=2), encoding="utf-8")
display(pd.Series(baseline_pointer, name="baseline_handoff"))
print("Next: run 04_model_selection.ipynb")


## Stop here before the final assessment

Notebook 03 is development work. It may be rerun while you are deciding the recipe. Notebook 04 is where the detailed study is frozen and the holdout is finally opened for the selected unchanged winner.
